In [1]:
import os
import pandas as pd
import numpy as np

# Set TTM
ttm = 27

# Create directory for output if it doesn't exist
output_dir = "RiskPremia/Tau-independent/unique/moneyness_step_0d01/Bitcoin_Premium"
os.makedirs(output_dir, exist_ok=True)

# Load data
daily_price = pd.read_csv("data/BTC_USD_Quandl_2011_2023.csv", parse_dates=["Date"])
daily_price = daily_price.sort_values("Date")

# Convert date format
daily_price["Date"] = daily_price["Date"].dt.strftime("%Y-%m-%d")

# Convert to numeric and sort
daily_price = daily_price.sort_values("Date")

daily_price["Adj.Close"] = pd.to_numeric(daily_price["Adj.Close"], errors='coerce')

# 27-day realized returns
S_t_plus_ttm = daily_price["Adj.Close"].iloc[ttm:].values
S_t = daily_price["Adj.Close"].iloc[:-ttm].values
daily_price["return_t_minus_27"] = np.concatenate(([np.nan] * 27, (S_t_plus_ttm - S_t) / S_t))

daily_price["return_t_plus_27"] = np.concatenate(((S_t_plus_ttm - S_t) / S_t, [np.nan] * 27))

# 1-day realized returns
S_t_plus_1 = daily_price["Adj.Close"].iloc[1:].values
S_t = daily_price["Adj.Close"].iloc[:-1].values
daily_price["return_t_minus_1"] = np.concatenate(([np.nan], (S_t_plus_1 - S_t) / S_t))

daily_price["return_t_plus_1"] = np.concatenate(((S_t_plus_1 - S_t) / S_t, [np.nan]))

# Drop NaN values
daily_price = daily_price.dropna()

# Filter dates
daily_price = daily_price[(daily_price["Date"] >= "2014-01-01") & (daily_price["Date"] <= "2022-12-31")]

# Calculate and display mean returns
print(daily_price["return_t_minus_27"].mean() * 365 / ttm)
print(daily_price["return_t_plus_27"].mean() * 365 / ttm)

# Save data
daily_price.to_excel(os.path.join(output_dir, "3287_sample_return_OA.xlsx"), index=False)

0.6793283466328047
0.6841611012242715
